In [0]:
'''# Widget Fallback for Secret Salt Handling
dbutils.widgets.text("hmac_secret_key", "EbayMarketplace2026SaltKey!", "HMAC Secret Salt")

def get_secret_salt():
    try:
        # Tries Databricks Secret Scope first
        return dbutils.secrets.get(scope="ebay_secrets", key="hmac_key")
    except Exception:
        # Falls back to widget parameter on Free Edition
        return dbutils.widgets.get("hmac_secret_key")

In [0]:
'''import datetime
import uuid
from pyspark.sql.functions import col, current_timestamp, lit, sha2, concat, expr, regexp_replace, to_timestamp, row_number
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, ArrayType, DecimalType, TimestampType
)

# 1. CATALOG & TABLE NAMES
CATALOG = "workspace"
SCHEMA = "ebay_pipeline"

TABLE_LOGS = f"{CATALOG}.{SCHEMA}.pipeline_execution_logs"
TABLE_BRONZE_FULL = f"{CATALOG}.{SCHEMA}.bronze_ebay_listings_full"
TABLE_BRONZE_INC = f"{CATALOG}.{SCHEMA}.bronze_ebay_listings_incremental"
TABLE_SILVER_LISTINGS = f"{CATALOG}.{SCHEMA}.silver_listings"
TABLE_SILVER_PRICE_HIST = f"{CATALOG}.{SCHEMA}.silver_price_history"
TABLE_SILVER_SELLERS = f"{CATALOG}.{SCHEMA}.silver_sellers"
TABLE_SILVER_CATEGORIES = f"{CATALOG}.{SCHEMA}.silver_categories"
TABLE_SILVER_QUARANTINE = f"{CATALOG}.{SCHEMA}.silver_quarantine"

# 2. EXPLICIT SCHEMA-ON-READ (BRONZE)
item_schema = StructType([
    StructField("itemId", StringType(), True),
    StructField("title", StringType(), True),
    StructField("categories", ArrayType(StructType([
        StructField("categoryId", StringType(), True),
        StructField("categoryName", StringType(), True)
    ])), True),
    StructField("price", StructType([
        StructField("value", StringType(), True),
        StructField("currency", StringType(), True)
    ]), True),
    StructField("condition", StringType(), True),
    StructField("buyingOptions", ArrayType(StringType()), True),
    StructField("seller", StructType([
        StructField("username", StringType(), True),
        StructField("feedbackScore", LongType(), True),
        StructField("feedbackPercentage", StringType(), True)
    ]), True),
    StructField("itemLocation", StructType([
        StructField("country", StringType(), True),
        StructField("stateOrProvince", StringType(), True),
        StructField("city", StringType(), True),
        StructField("postalCode", StringType(), True)
    ]), True),
    StructField("itemCreationDate", StringType(), True),
    StructField("itemWebUrl", StringType(), True)
])

raw_response_schema = StructType([
    StructField("total", LongType(), True),
    StructField("itemSummaries", ArrayType(item_schema), True)
])

# 3. HELPER FUNCTIONS
def get_secret_salt():
    try:
        return dbutils.secrets.get(scope="ebay_secrets", key="hmac_key")
    except:
        return "DEFAULT_HMAC_SALT_KEY_FALLBACK"

In [0]:
# Databricks notebook source
# COMMAND ----------
import os
import uuid
from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType, 
    ArrayType, DoubleType
)
from pyspark.sql import functions as F

# 1. CATALOG & SCHEMA SETUP
spark.sql("CREATE SCHEMA IF NOT EXISTS ebay_pipeline")
spark.sql("USE SCHEMA ebay_pipeline")

SCHEMA = "ebay_pipeline"
TABLE_LOGS = f"{SCHEMA}.pipeline_execution_logs"
TABLE_BRONZE_FULL = f"{SCHEMA}.bronze_ebay_listings_full"
TABLE_BRONZE_INC = f"{SCHEMA}.bronze_ebay_listings_incremental"
TABLE_SILVER_LISTINGS = f"{SCHEMA}.silver_listings"
TABLE_QUARANTINE = f"{SCHEMA}.silver_quarantine_records"

# 2. FILE PATHS (Dynamic workspace relative root)
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))) if "__file__" in locals() else "/Workspace/Users/ammaraa5688@gmail.com/Online-retail-data-pipeline"
RAW_FULL_DIR = f"{REPO_ROOT}/data/raw/full_load"
RAW_INC_DIR = f"{REPO_ROOT}/data/raw/incremental_load"

# 3. INITIALIZE AUDIT TABLE (No custom LOCATION parameter)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_LOGS} (
    log_id STRING,
    layer STRING,
    table_name STRING,
    parameter_processed STRING,
    execution_start_time TIMESTAMP,
    execution_end_time TIMESTAMP,
    status STRING,
    rows_inserted BIGINT,
    rows_updated BIGINT,
    error_message STRING
)
USING DELTA
""")

# 4. AUDIT LOGGER HELPER
audit_log_schema = StructType([
    StructField("log_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("parameter_processed", StringType(), True),
    StructField("execution_start_time", TimestampType(), False),
    StructField("execution_end_time", TimestampType(), True),
    StructField("status", StringType(), False),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("error_message", StringType(), True)
])

def log_pipeline_run(layer, table_name, param, start_time, end_time, status, inserted=0, updated=0, error=None):
    entry = [(
        str(uuid.uuid4()),
        layer,
        table_name,
        str(param),
        start_time,
        end_time,
        status,
        int(inserted) if inserted is not None else 0,
        int(updated) if updated is not None else 0,
        str(error) if error else None
    )]
    df_log = spark.createDataFrame(entry, schema=audit_log_schema)
    df_log.write.format("delta").mode("append").saveAsTable(TABLE_LOGS)